# IMU Interfacing
Please read the `README.md` file before running this tutorial.

For this tutorial, you will need to have at least one sequence from the Boreas Road Trip (Boreas-RT) dataset downloaded. Ideally, the sequence should have Aeva data (see the `aeva_split` from `pyboreas/data/metadata_splits`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from pyboreas import BoreasDataset
from pyboreas.utils.utils import get_inverse_tf

root = '/path/to/data/boreas/'
split = None
# AWS: Note: Free Tier SageMaker instances don't have enough storage (25 GB) for 1 sequence (100 GB)
# root = '/home/ec2-user/SageMaker/boreas/'
# split = [['boreas-2021-09-02-11-42', 163059759e6, 163059760e6-1]]

# With verbose=True, the following will print information about each sequence
# Note that we need to specify that we want to load in auxiliary frames (IMUs, encoder) since they take up time to load and are not needed for all applications
bd = BoreasDataset(root, split=split, verbose=True, load_aux_frames=True)
# Grab the first sequence
seq = bd.sequences[0]

Two types of IMUs can be interfaced with using pyboreas with Boreas-RT sequences:

1. the DMU41 IMU
2. the Aeva built-in IMU for sequences with Aeva data available

Note that if your chosen sequence doesn't have Aeva data, then there will be 0 Aeva frames and 0 Aeva IMU frames available for your sequence (see print-out above). The calibration file is still included however.

In [ ]:
# We can access the calibration data for each sensor as follows:
T_applanix_dmu = seq.calib.T_applanix_dmu
T_applanix_aeva = seq.calib.T_applanix_lidar @ get_inverse_tf(seq.calib.T_aeva_lidar)

print('T_applanix_dmu:\n', T_applanix_dmu)
print('T_applanix_aeva:\n', T_applanix_aeva)

We can load IMU data through seq.dmu or seq.aeva_imu either

1. all at once
2. by range
3. iteratively

In [ ]:
# All at once loading example
# We can access all data at once using dmu_data (or aeva_data)
dmu = seq.get_dmu_data()
dmu_timestamps_micro = dmu['timestamps_micro']
dmu_body_angvel = dmu['body_angvel']
dmu_body_acc = dmu['body_acc']

# Plot angular velocity as an example
dmu_timestamps_sec = (dmu_timestamps_micro - dmu_timestamps_micro[0]) * 1e-6  # make the first timestamp 0 for easier plotting

fig = plt.figure(figsize=(10, 5))
plt.plot(dmu_timestamps_sec, dmu_body_angvel[:, 0], label='body_angvel_x')
plt.plot(dmu_timestamps_sec, dmu_body_angvel[:, 1], label='body_angvel_y')
plt.plot(dmu_timestamps_sec, dmu_body_angvel[:, 2], label='body_angvel_z')
plt.xlabel('Time (s)')
plt.ylabel('Body Angular Velocity (rad/s)')
plt.title('DMU Body Angular Velocity')
plt.legend()
plt.grid()

In [ ]:
# By range loading example
# We can use the get_dmu_data method to load data in a specific time range (in microseconds)
# For example, let's load all IMU measurements between the first two radar frames
radar_frame_0 = seq.get_radar(0)
radar_frame_1 = seq.get_radar(1)
start_micro = radar_frame_0.timestamp_micro
end_micro = radar_frame_1.timestamp_micro

# Load only DMU measurements in this time range
dmu_range = seq.get_dmu_data(start_micro, end_micro)
dmu_range_timestamps_micro = dmu_range['timestamps_micro']
dmu_range_body_angvel = dmu_range['body_angvel']
dmu_range_body_acc = dmu_range['body_acc']

# We expect to load around 50 DMU measurements (if no drop-outs) since the DMU runs at 200 Hz and the radar runs at 4 Hz
print(f'Loaded {len(dmu_range_timestamps_micro)} DMU measurements between radar frame 0 and 1')

In [ ]:
# Iteratively loading example
# Note that unloading data is not as important as in the camera/lidar/radar case since IMU data is very light, but the option exists for maximum memory efficiency.
max_iter = 3

for idx, dmu_frame in enumerate(seq.dmu):
    print(f'Iteratively loaded DMU frame {idx}')
    print('DMU timestamp (microseconds):', dmu_frame.timestamp_micro)
    print('DMU body angular velocity (rad/s):', dmu_frame.body_angvel)
    print('DMU body acceleration (m/s^2):', dmu_frame.body_acc)
    print('---')
    dmu_frame.unload_data() # Free memory by unloading data from the frame
    if idx >= max_iter - 1:
        break

## IMU alignment example
This example requires both DMU and Aeva data (must be a Sequence with Aeva/FMCW Lidar available). We show how to use the extrinsic transforms to align the two IMU's to (arbitrarily) the lidar frame.

Note: the Aeva IMU has a small, but notable, bias unlike the DMU41. You will likely see a small offset between the two sensors as a result of this.

In [ ]:
if not seq.has_aeva:
    print('This sequence does not have Aeva IMU data, skipping Aeva IMU examples.')
else:
    # Load in all of Aeva IMU data at once
    aeva_imu = seq.get_aeva_imu_data()
    aeva_imu_timestamps_micro = aeva_imu['timestamps_micro']
    aeva_imu_body_angvel = aeva_imu['body_angvel']
    aeva_imu_body_acc = aeva_imu['body_acc']

    # Make first timestamp 0 from the start of the DMU data for easier plotting
    aeva_imu_timestamps_sec = (aeva_imu_timestamps_micro - dmu_timestamps_micro[0]) * 1e-6

    # Load in calibration between Aeva IMU and lidar
    T_lidar_aeva_imu = get_inverse_tf(seq.calib.T_imu_aeva @ seq.calib.T_aeva_lidar)
    print('T_lidar_aeva_imu:\n', T_lidar_aeva_imu)

    # Load in calibration between DMU and lidar
    T_lidar_dmu = get_inverse_tf(seq.calib.T_applanix_lidar) @ seq.calib.T_applanix_dmu
    print('T_lidar_dmu:\n', T_lidar_dmu)

    # Transform angular velocities into lidar frame from both IMU sources
    aeva_imu_angvel_lidar = (T_lidar_aeva_imu[:3, :3] @ aeva_imu_body_angvel.T).T
    dmu_angvel_lidar = (T_lidar_dmu[:3, :3] @ dmu_body_angvel.T).T

    # Plot Aeva IMU and DMU angular velocities in the lidar frame for comparison
    fig = plt.figure(figsize=(10, 5))
    plt.subplot(3, 1, 1)
    plt.plot(aeva_imu_timestamps_sec, aeva_imu_angvel_lidar[:, 0], label='Aeva IMU')
    plt.plot(dmu_timestamps_sec, dmu_angvel_lidar[:, 0], label='DMU')
    plt.ylabel('Ang. Vel. X (rad/s)')
    plt.title('Aeva IMU vs DMU Angular Velocity in Lidar Frame')
    plt.legend()
    plt.grid()
    plt.subplot(3, 1, 2)
    plt.plot(aeva_imu_timestamps_sec, aeva_imu_angvel_lidar[:, 1], label='Aeva IMU')
    plt.plot(dmu_timestamps_sec, dmu_angvel_lidar[:, 1], label='DMU')
    plt.ylabel('Ang. Vel. Y (rad/s)')
    plt.grid()
    plt.subplot(3, 1, 3)
    plt.plot(aeva_imu_timestamps_sec, aeva_imu_angvel_lidar[:, 2], label='Aeva IMU')
    plt.plot(dmu_timestamps_sec, dmu_angvel_lidar[:, 2], label='DMU')
    plt.xlabel('Time (s)')
    plt.ylabel('Ang. Vel. Z (rad/s)')
    plt.grid()
    plt.show()
